## Data Exploration

In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px
import matplotlib.pyplot as plt
import scipy.stats as sp

In [ ]:
import os
os.chdir(r"C:\Users\belle\OneDrive\Personal Projects\athlete_profiles\athlete_clustering")

In [4]:
df = pd.read_csv(r"data\Longitudinal_Athlete_Sports_Performance_RL_Dataset.csv")  

Explore data.

In [5]:
df.head(3).T

,0,1,2
Athlete_ID,1,1,1
Session_ID,1,2,3
Days_From_Start,0,2,4
Sport,Athletics,Swimming,Swimming
Previous_Performance,64.301043,59.949164,56.71246
Previous_Training_Load,462.694009,462.694009,432.009171
Resting_Heart_Rate_bpm,65.852008,67.917641,64.709823
Heart_Rate_Variability_ms,54.573129,47.701078,68.285953
Oxygen_Saturation_pct,97.245654,98.350685,97.342644
Body_Temperature_C,36.566646,36.623072,36.582591


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Athlete_ID                   10000 non-null  int64  
 1   Session_ID                   10000 non-null  int64  
 2   Days_From_Start              10000 non-null  int64  
 3   Sport                        10000 non-null  str    
 4   Previous_Performance         10000 non-null  float64
 5   Previous_Training_Load       10000 non-null  float64
 6   Resting_Heart_Rate_bpm       10000 non-null  float64
 7   Heart_Rate_Variability_ms    9900 non-null   float64
 8   Oxygen_Saturation_pct        10000 non-null  float64
 9   Body_Temperature_C           10000 non-null  float64
 10  Sleep_Duration_hr            10000 non-null  float64
 11  Sleep_Quality_Score          9900 non-null   float64
 12  Rest_Duration_hr             10000 non-null  float64
 13  Training_Load               

In [7]:
df.describe()

,Athlete_ID,Session_ID,Days_From_Start,Previous_Performance,Previous_Training_Load,Resting_Heart_Rate_bpm,Heart_Rate_Variability_ms,Oxygen_Saturation_pct,Body_Temperature_C,Sleep_Duration_hr,...,Training_Action_ID,Training_Intensity_pct,Training_Duration_min,Training_Volume,Next_Performance,Next_Fatigue_Score,Next_Recovery_Score,Performance_Gain,Training_Efficiency,Multi_Objective_Reward
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,9900.000000,10000.000000,10000.000000,10000.000000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,125.500000,20.500000,39.000000,68.801201,454.012940,65.282241,61.980308,97.170984,36.700628,7.203404,...,2.047100,57.228997,88.021688,51.095933,55.437999,53.376132,59.321974,4.282937,0.088428,2.338819
std,72.171815,11.543974,23.087947,8.783047,109.745419,3.882605,5.640111,0.803723,0.195794,0.900977,...,1.358406,9.422707,13.928524,14.804109,7.579011,9.677702,8.294836,3.119522,0.071695,2.068615
min,1.000000,1.000000,0.000000,32.438981,100.000000,48.859919,39.322333,94.420629,36.100000,4.500000,...,0.000000,20.951963,39.352630,11.002526,26.967617,20.320169,27.623254,-8.271482,-0.355060,-7.483354
25%,63.000000,10.750000,19.500000,62.782876,379.621009,62.657939,58.181757,96.627472,36.567152,6.591288,...,1.000000,50.884309,78.592501,40.453693,50.289450,46.950252,53.679256,2.197144,0.042861,0.964321
50%,125.500000,20.500000,39.000000,68.925588,453.863907,65.293555,61.944094,97.162011,36.698276,7.202063,...,2.000000,57.245179,88.065883,49.987970,55.468375,53.346538,59.419252,4.281885,0.083805,2.375097
75%,188.000000,30.250000,58.500000,74.855625,527.210960,67.859634,65.750753,97.716513,36.833364,7.818613,...,3.000000,63.708636,97.528082,60.359811,60.551476,59.959262,64.903274,6.392879,0.129432,3.738758
max,250.000000,40.000000,78.000000,98.547643,841.199906,81.787685,84.843426,100.000000,37.380135,9.500000,...,4.000000,90.178996,150.000000,128.564656,82.087935,91.569899,94.012243,16.736087,0.492888,10.186557


Summary stats.

Traits of clusters we might be interested in: 

* Overtraining
* Underperforming
* High natural ability
* Weakness in [speed/strength/endurance/recovery]
* Improving or not

Features to include to capture this:

* Average resting heart rate
* Average speed
* Average strength
* Average endurance
* Change in resting heart rate
* Average perfomance - training load ratio
* Average training load
* Average recovery - training load ratio
* Average performance - recovery ratio
* Change in performance
* Average change in performance


In [35]:
summary = df.groupby(['Athlete_ID']).agg({
    'Athlete_ID': 'count',
    'Sport': 'nunique',
    'Resting_Heart_Rate_bpm': 'mean',
 #   'Speed_Score': 'mean',
 #   'Strength_Score': 'mean',
    'Training_Load': 'mean',
 #   'Endurance_Score': 'mean',
    'Training_Frequency_per_Week': 'mean',
    'Current_Performance': 'mean'
})
summary

,Athlete_ID,Sport,Resting_Heart_Rate_bpm,Training_Load,Training_Frequency_per_Week,Current_Performance
Athlete_ID,,,,,,
1,40,5,65.772410,464.131621,4.987120,46.182948
2,40,5,65.206532,451.451094,4.943073,49.796008
3,40,5,65.392055,439.117259,5.017330,51.724831
4,40,5,65.049863,453.578849,5.022981,50.004490
5,40,5,65.904176,431.143973,5.133898,49.414633
...,...,...,...,...,...,...
246,40,5,64.898869,450.447899,5.156643,47.770982
247,40,5,65.752937,435.636603,4.652908,58.888550
248,40,5,64.632669,444.076951,4.976780,53.508240


In [36]:
fig = px.scatter_matrix(summary, 
                  dimensions=['Resting_Heart_Rate_bpm', 
                              #'Speed_Score', 
                              #'Strength_Score', 
                              'Training_Load', 
                              #'Endurance_Score', 
                              'Training_Frequency_per_Week', 'Current_Performance'], 
                  title='Scatter Matrix of Athlete Summary Metrics by Sport',
                  height=800, width=800, opacity=0.7)
fig.update_layout(font=dict(size=6))
fig.show()

In [37]:
summary.to_csv(r"data\athlete_summary.csv", index=False)